# KNN Outliers

The k-nearest-neighbour (KNN) detector scores each observation by its distance to its nearest neighbours. Points far from all others get high scores. It needs no training and makes no distributional assumption.

In [ ]:
Pollis.packages("OutlierDetection", "OutlierDetectionNeighbors", "Plots")

In [ ]:
using OutlierDetection, OutlierDetectionNeighbors
using Plots, Statistics, Random

Random.seed!(1);

## Data

Two hundred points from a normal cloud, plus eight points scattered far from it. `is_true` marks the planted outliers so we can check the detector.

In [ ]:
far = [5.0 5.0; -5.0 5.0; 5.0 -5.0; -5.0 -5.0; 5.0 0.0; -5.0 0.0; 0.0 5.0; 0.0 -5.0]
X = vcat(randn(200, 2), far .+ 0.5 .* randn(8, 2))
is_true = vcat(falses(200), trues(8))

scatter(X[:, 1], X[:, 2]; group = ifelse.(is_true, "Planted outlier", "Normal"),
    markerstrokewidth = 0, xlabel = "Feature 1", ylabel = "Feature 2", title = "Data")

## Fit the Detector

`OutlierDetection.fit` expects observations in **columns**, so the matrix is transposed. With `reduction = :maximum` the score is the distance to the k-th nearest neighbour.

In [ ]:
detector = KNNDetector(k = 10, reduction = :maximum)
model, scores = OutlierDetection.fit(detector, permutedims(X); verbosity = 0)

scatter(X[:, 1], X[:, 2]; zcolor = scores, markerstrokewidth = 0, colorbar_title = "Score",
    xlabel = "Feature 1", ylabel = "Feature 2", title = "KNN scores (k = 10)", label = "")

## Flag the Top Scores

With a contamination of 5% we flag the top 5% of scores and compare with the planted outliers.

In [ ]:
threshold = quantile(scores, 1 - 0.05)
flagged = scores .> threshold

println("Flagged: ", sum(flagged))
println("Planted outliers found: ", sum(flagged .& is_true), " of ", sum(is_true))

## Score Distribution

A clear right tail separated from the bulk means the outliers stand apart.

In [ ]:
histogram(scores; bins = 40, xlabel = "KNN score", ylabel = "Count", label = "")
vline!([threshold]; color = :red, linestyle = :dash, label = "Top 5%")

## Choice of k and Reduction

Small k reacts to single isolated points; larger k also catches small groups. The `:mean` reduction averages over all k neighbours and is smoother. The top-ranked points should be stable across reasonable settings.

In [ ]:
for k in (5, 10, 20), red in (:maximum, :mean)
    _, s = OutlierDetection.fit(KNNDetector(k = k, reduction = red), permutedims(X); verbosity = 0)
    top = s .> quantile(s, 0.95)
    println(rpad("k = $k, $red", 18), "planted found: ", sum(top .& is_true))
end

## Summary
- KNN scores a point by its distance to its k-th (or mean of k) nearest neighbours.
- Transpose the data: OutlierDetection.jl expects observations in columns.
- Flag the top share of scores and check the result for several values of k.
- KNN uses one global distance scale; for clusters of different density, use LOF.